# Fine Tuning with HF Transformers

In [ ]:
from __future__ import annotations

from datasets import load_dataset
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DataCollatorForCompletionOnlyLM, SFTConfig, SFTTrainer

MODEL_LOC = "../model/Llama-3.2-1B-Instruct"
MADE_MODEL_LOC = "../model/FT-Llama-3.2-1B-Instruct"

In [ ]:
def format_chat_message(example):
    """This function formats the chat message for the user.

    Args:
        example (dict): A dictionary containing the sentence, event type, event
            trigger, subject, and object.

    Returns:
        dict: A dictionary containing the formatted chat message.
    """
    event_types = (
        "Build",
        "Clean",
        "Cook",
        "Exercise",
        "Hire",
        "Injure",
        "Meet",
        "Purchase",
        "Transport",
        "Travel",
    )
    question = (
        f"I am going to give you a sentence and I want you to tell me what type "
        "of event it is describing and what is the indicating verb, object, "
        "and subject of the sentence. There are ten possible types of events "
        "you can choose from. "
        "Please give your answer in a JSON with the key 'EventType' matching "
        f"one of the following options: {event_types}. The 'EventTrigger' "
        "matching the relevant verb in the sentence. The 'Subject' matching "
        "the actor in the sentence, and 'Object' matching the target object.\n"
        f"The sentence is: {example['Sentence']}"
    )
    answer = (
        f"{{'EventType': '{example['EventType']}', "
        f"'EventTrigger': '{example['EventTrigger']}', "
        f"'Subject': '{example['Subject']}', "
        f"'Object': '{example['Object']}'}}"
    )
    return {
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ]
    }

In [ ]:
# Load the dataset

dataset = load_dataset(
    "csv",
    data_files={"train": "../data/gpt_train.tsv", "eval": "../data/gpt_eval.tsv"},
    delimiter="\t",
)

formatted_dataset = dataset.map(
    format_chat_message, remove_columns=dataset["train"].column_names
)

In [ ]:
# Load the model!

tokenizer = AutoTokenizer.from_pretrained(MODEL_LOC)
model = AutoModelForCausalLM.from_pretrained(MODEL_LOC)

responce_template = "<|start_header_id|>assistant<|end_header_id|>"
collator = DataCollatorForCompletionOnlyLM(responce_template, tokenizer=tokenizer)
tokenizer.pad_token = "<|finetune_right_pad_id|>"

In [ ]:
training_args = SFTConfig(
    # packing=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    logging_dir="../logs",
    logging_steps=30,
    do_eval=True,
    eval_strategy="steps",
    output_dir="../model/output",
    learning_rate=1e-5,
    weight_decay=0.1,
    # neftune_noise_alpha=1,
)

peft_config = LoraConfig(
    r=4,
    lora_alpha=4,
    lora_dropout=0.1,
    use_rslora=True,
    bias="none",
    # init_lora_weights="pissa",
    task_type="CAUSAL_LM",
    target_modules=["k_proj", "v_proj", "up_proj"],
    # target_modules=[
    # "q_proj", "k_proj", "v_proj", "up_proj", "down_proj", "o_proj", "gate_proj"
    # ],
)


trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["eval"],
    peft_config=peft_config,
    data_collator=collator,
)

In [ ]:
trainer.train()

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(MODEL_LOC)
fine_tuned_model = PeftModel.from_pretrained(
    base_model, "../model/output/checkpoint-180"
)

merged_model = fine_tuned_model.merge_and_unload()

merged_model.save_pretrained(MADE_MODEL_LOC, save_tokenizer=True)